# 03 · Preprocessing & Feature Engineering

**Project:** Credit Default Risk — AI Governance Portfolio (Notebook 3 of the pipeline)

---

## Goal

Turn 10 raw tables into **one model-ready matrix**, applying the fixes that `01`/`02`
identified — while keeping the pipeline **leakage-free and auditable**.

Scope decision: a **moderate, explainability-first** feature set (aggregate only the *core*
signals from each auxiliary table). A high-risk credit model under the **EU AI Act** must be
explainable and its features defensible, so we prefer ~dozens of named, documented features
over hundreds of opaque ones.

### Steps
1. **Aggregate** the 6 auxiliary tables to **one row per applicant** (`SK_ID_CURR`).
2. **Clean** the main table (sentinel → NaN + flag, drop high-missing block, engineer ratios).
3. **Join** the aggregates onto the main table.
4. **Set aside the protected attributes** (raw) for the Fair Lending audit.
5. **Encode** categoricals, **split** (stratified), **impute + scale** — *fit on train only*.
6. **Save** the processed matrices and run **governance checks**.


## 0 · Setup

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split   # stratified split
from sklearn.impute import SimpleImputer                # median imputation
from sklearn.preprocessing import StandardScaler        # feature scaling

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

DATA_DIR = "../data"
PROC_DIR = "../data/processed"          # processed matrices land here (git-ignored)
os.makedirs(PROC_DIR, exist_ok=True)

SENTINEL = 365243                       # DAYS_EMPLOYED placeholder found in 02
print("Setup complete.")

Setup complete.


## 1 · Aggregate the auxiliary tables to one row per applicant

Each auxiliary table has **many rows per applicant** (monthly histories, multiple prior
loans). We collapse each to **one row keyed by `SK_ID_CURR`** using a small, *named* set of
aggregates per table. Big tables are loaded **one at a time and freed** to respect 16 GB RAM.


In [2]:
def agg_bureau(data_dir):
    "Aggregate credit-bureau history (bureau + bureau_balance) to one row per SK_ID_CURR."
    # --- bureau_balance: monthly status rows per bureau credit ---
    bb = pd.read_csv(os.path.join(data_dir, "bureau_balance.csv"))
    # STATUS in {1..5} means the credit was that many months past due; flag it.
    bb["DPD"] = bb["STATUS"].isin(["1", "2", "3", "4", "5"]).astype(int)
    bb_agg = bb.groupby("SK_ID_BUREAU").agg(
        BB_MONTHS=("MONTHS_BALANCE", "count"),   # how many monthly records exist
        BB_DPD=("DPD", "sum"),                   # how many of those months were past due
    )
    del bb                                       # free 27M-row table immediately

    # --- bureau: one row per external credit; attach the bureau_balance summary ---
    bureau = pd.read_csv(os.path.join(data_dir, "bureau.csv"))
    bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
    del bb_agg
    bureau["IS_ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)

    agg = bureau.groupby("SK_ID_CURR").agg(
        BUREAU_COUNT=("SK_ID_BUREAU", "count"),               # # of bureau credits
        BUREAU_ACTIVE_COUNT=("IS_ACTIVE", "sum"),             # # currently active
        BUREAU_AMT_CREDIT_SUM_mean=("AMT_CREDIT_SUM", "mean"),# avg credit amount
        BUREAU_AMT_CREDIT_SUM_sum=("AMT_CREDIT_SUM", "sum"),  # total credit amount
        BUREAU_AMT_DEBT_sum=("AMT_CREDIT_SUM_DEBT", "sum"),   # total current debt
        BUREAU_OVERDUE_DAYS_max=("CREDIT_DAY_OVERDUE", "max"),# worst days overdue
        BUREAU_AMT_OVERDUE_sum=("AMT_CREDIT_SUM_OVERDUE", "sum"), # total overdue amount
        BUREAU_DAYS_CREDIT_mean=("DAYS_CREDIT", "mean"),      # avg age of credits
        BUREAU_DAYS_CREDIT_min=("DAYS_CREDIT", "min"),        # oldest credit (recency)
        BUREAU_PROLONG_sum=("CNT_CREDIT_PROLONG", "sum"),     # # of prolongations
        BUREAU_BB_MONTHS_mean=("BB_MONTHS", "mean"),          # avg monthly history length
        BUREAU_BB_DPD_sum=("BB_DPD", "sum"),                  # total past-due months
    )
    del bureau
    return agg

print("agg_bureau defined")

agg_bureau defined


In [3]:
def agg_previous(data_dir):
    "Aggregate previous Home Credit applications to one row per SK_ID_CURR."
    prev = pd.read_csv(os.path.join(data_dir, "previous_application.csv"))
    prev["IS_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
    prev["IS_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)

    agg = prev.groupby("SK_ID_CURR").agg(
        PREV_COUNT=("SK_ID_PREV", "count"),                       # # of previous applications
        PREV_APPROVED_COUNT=("IS_APPROVED", "sum"),               # # approved
        PREV_REFUSED_COUNT=("IS_REFUSED", "sum"),                 # # refused
        PREV_AMT_APPLICATION_mean=("AMT_APPLICATION", "mean"),    # avg amount asked
        PREV_AMT_CREDIT_mean=("AMT_CREDIT", "mean"),              # avg amount granted
        PREV_AMT_DOWN_PAYMENT_mean=("AMT_DOWN_PAYMENT", "mean"),  # avg down payment
        PREV_RATE_DOWN_PAYMENT_mean=("RATE_DOWN_PAYMENT", "mean"),# avg down-payment rate
        PREV_DAYS_DECISION_mean=("DAYS_DECISION", "mean"),        # avg recency of decisions
        PREV_CNT_PAYMENT_mean=("CNT_PAYMENT", "mean"),            # avg term length
    )
    # Refusal RATE is more informative than the raw count -> engineer it.
    agg["PREV_REFUSED_RATE"] = agg["PREV_REFUSED_COUNT"] / agg["PREV_COUNT"]
    del prev
    return agg

print("agg_previous defined")

agg_previous defined


In [4]:
def agg_pos(data_dir):
    "Aggregate POS / cash-loan monthly balances to one row per SK_ID_CURR."
    pos = pd.read_csv(os.path.join(data_dir, "POS_CASH_balance.csv"))
    agg = pos.groupby("SK_ID_CURR").agg(
        POS_MONTHS_COUNT=("MONTHS_BALANCE", "count"),   # length of POS history
        POS_SK_DPD_mean=("SK_DPD", "mean"),             # avg days past due
        POS_SK_DPD_max=("SK_DPD", "max"),               # worst days past due
        POS_SK_DPD_DEF_mean=("SK_DPD_DEF", "mean"),     # avg days past due (with tolerance)
    )
    del pos
    return agg

def agg_installments(data_dir):
    "Aggregate repayment history to one row per SK_ID_CURR."
    inst = pd.read_csv(os.path.join(data_dir, "installments_payments.csv"))
    # How many days late each payment was (negative = early -> clip to 0).
    inst["DPD"] = (inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]).clip(lower=0)
    # How much was underpaid vs scheduled (positive = paid less than due).
    inst["PAYMENT_DIFF"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]
    # Fraction of the scheduled amount actually paid.
    inst["PAYMENT_RATIO"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)
    agg = inst.groupby("SK_ID_CURR").agg(
        INST_COUNT=("DAYS_INSTALMENT", "count"),            # # of installments paid
        INST_DPD_mean=("DPD", "mean"),                      # avg lateness
        INST_DPD_max=("DPD", "max"),                        # worst lateness
        INST_PAYMENT_DIFF_mean=("PAYMENT_DIFF", "mean"),    # avg underpayment
        INST_PAYMENT_RATIO_mean=("PAYMENT_RATIO", "mean"),  # avg paid-vs-due ratio
    )
    del inst
    return agg

def agg_creditcard(data_dir):
    "Aggregate credit-card monthly balances to one row per SK_ID_CURR."
    cc = pd.read_csv(os.path.join(data_dir, "credit_card_balance.csv"))
    # Utilization = balance / credit limit (a classic risk signal).
    cc["UTILIZATION"] = cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
    agg = cc.groupby("SK_ID_CURR").agg(
        CC_MONTHS_COUNT=("MONTHS_BALANCE", "count"),        # length of card history
        CC_AMT_BALANCE_mean=("AMT_BALANCE", "mean"),        # avg balance carried
        CC_UTILIZATION_mean=("UTILIZATION", "mean"),        # avg utilization
        CC_SK_DPD_mean=("SK_DPD", "mean"),                  # avg days past due
        CC_SK_DPD_max=("SK_DPD", "max"),                    # worst days past due
        CC_DRAWINGS_mean=("AMT_DRAWINGS_CURRENT", "mean"),  # avg monthly drawings
    )
    del cc
    return agg

print("agg_pos / agg_installments / agg_creditcard defined")

agg_pos / agg_installments / agg_creditcard defined


In [5]:
# Run all six aggregations one at a time (each frees its big table before the next).
aggregations = {}
for name, fn in [("bureau", agg_bureau), ("previous", agg_previous), ("pos", agg_pos),
                 ("installments", agg_installments), ("creditcard", agg_creditcard)]:
    aggregations[name] = fn(DATA_DIR)
    print(f"{name:14s} -> {aggregations[name].shape[1]} features, "
          f"{aggregations[name].shape[0]:,} applicants covered")

total_agg_features = sum(a.shape[1] for a in aggregations.values())
print(f"\nTotal aggregated features: {total_agg_features}")

bureau         -> 12 features, 305,811 applicants covered


previous       -> 10 features, 338,857 applicants covered


pos            -> 4 features, 337,252 applicants covered


installments   -> 5 features, 339,587 applicants covered


creditcard     -> 6 features, 103,558 applicants covered

Total aggregated features: 37


## 2 · Clean the main table & engineer ratio features

Apply the `02` findings to `application_train`:
- replace the `DAYS_EMPLOYED` sentinel with `NaN` **and keep a flag**,
- drop the high-missing **housing/building stats** block (`*_AVG/_MODE/_MEDI`),
- collapse the 20 sparse `FLAG_DOCUMENT_*` columns into a single count,
- engineer a few **ratio features** that encode affordability.


In [6]:
app = pd.read_csv(os.path.join(DATA_DIR, "application_train.csv"))
start_cols = app.shape[1]

# (a) DAYS_EMPLOYED sentinel -> NaN, but preserve the "no employment record" signal as a flag.
app["DAYS_EMPLOYED_ANOMALY"] = (app["DAYS_EMPLOYED"] == SENTINEL).astype(int)
app["DAYS_EMPLOYED"] = app["DAYS_EMPLOYED"].replace(SENTINEL, np.nan)

# (b) Drop the housing/building statistics block (47 columns, ~50-70% missing).
building_cols = [c for c in app.columns if c.endswith(("_AVG", "_MODE", "_MEDI"))]
app = app.drop(columns=building_cols)

# (c) Collapse the 20 document flags into one count (most are near-constant individually).
doc_cols = [c for c in app.columns if c.startswith("FLAG_DOCUMENT")]
app["DOCUMENT_SUM"] = app[doc_cols].sum(axis=1)
app = app.drop(columns=doc_cols)

# (d) Engineer affordability ratios (interpretable, governance-friendly features).
app["CREDIT_INCOME_RATIO"] = app["AMT_CREDIT"] / app["AMT_INCOME_TOTAL"]      # loan size vs income
app["ANNUITY_INCOME_RATIO"] = app["AMT_ANNUITY"] / app["AMT_INCOME_TOTAL"]    # yearly payment vs income
app["CREDIT_GOODS_RATIO"] = app["AMT_CREDIT"] / app["AMT_GOODS_PRICE"]        # loan vs price of good
app["EMPLOYED_TO_AGE"] = app["DAYS_EMPLOYED"] / app["DAYS_BIRTH"]             # tenure relative to age

# (e) CODE_GENDER 'XNA' (4 rows) is not a real category -> treat as missing.
app["CODE_GENDER"] = app["CODE_GENDER"].replace("XNA", np.nan)

print(f"Main table columns: {start_cols} -> {app.shape[1]} "
      f"(dropped {len(building_cols)} building + {len(doc_cols)} document, added 6 engineered)")

Main table columns: 122 -> 61 (dropped 47 building + 20 document, added 6 engineered)


## 3 · Join the aggregates onto the main table

In [7]:
# Left-join every aggregation on SK_ID_CURR (keep all applicants, even those with no history).
df = app.set_index("SK_ID_CURR")
for name, agg in aggregations.items():
    df = df.join(agg, how="left")        # applicants missing from an aux table -> NaN
df = df.reset_index()

# Count-type aggregates are meaningfully ZERO (not missing) when an applicant has no history.
count_cols = [c for c in df.columns if c.endswith("_COUNT") or c.endswith("_count")
              or c in ("BUREAU_BB_DPD_sum", "BUREAU_PROLONG_sum")]
df[count_cols] = df[count_cols].fillna(0)

print("Joined matrix shape:", df.shape)
print("Example aggregated columns now present:",
      [c for c in df.columns if c.startswith(("BUREAU_", "PREV_", "INST_"))][:6])

Joined matrix shape: (307511, 98)
Example aggregated columns now present: ['BUREAU_COUNT', 'BUREAU_ACTIVE_COUNT', 'BUREAU_AMT_CREDIT_SUM_mean', 'BUREAU_AMT_CREDIT_SUM_sum', 'BUREAU_AMT_DEBT_sum', 'BUREAU_OVERDUE_DAYS_max']


## 4 · Set aside the protected attributes (raw) for the audit

Before encoding, we **snapshot the protected attributes in their original, human-readable
form**, keyed by `SK_ID_CURR`. The model will be trained on encoded features, but the
Fair Lending audit in `05` needs the *raw* groups to slice predictions by — so we persist
them separately and never let them get lost in encoding.


In [8]:
# Derive readable age bands (age is a protected basis).
age_years = -df["DAYS_BIRTH"] / 365.25
protected = pd.DataFrame({
    "SK_ID_CURR": df["SK_ID_CURR"],
    "CODE_GENDER": df["CODE_GENDER"],
    "NAME_FAMILY_STATUS": df["NAME_FAMILY_STATUS"],
    "NAME_EDUCATION_TYPE": df["NAME_EDUCATION_TYPE"],
    "AGE_BAND": pd.cut(age_years, [20, 30, 40, 50, 60, 100],
                       labels=["20s", "30s", "40s", "50s", "60+"]),
    "CNT_CHILDREN": df["CNT_CHILDREN"],
})
print("Protected-attribute snapshot:", protected.shape)
protected.head(3)

Protected-attribute snapshot: (307511, 6)


,SK_ID_CURR,CODE_GENDER,NAME_FAMILY_STATUS,NAME_EDUCATION_TYPE,AGE_BAND,CNT_CHILDREN
0,100002,M,Single / not married,Secondary / secondary special,20s,0
1,100003,F,Married,Higher education,40s,0
2,100004,M,Single / not married,Secondary / secondary special,50s,0


## 5 · Encode → split → impute → scale (leakage-free)

**Order matters for leakage.** We split **first**, then fit the imputer and scaler on the
**training set only** and apply those fitted transforms to validation. Fitting on the full
data before splitting would leak validation statistics into training — a real audit finding.

Categoricals are **label-encoded** (one integer column each) to keep the feature count
moderate and the matrix tree-model-friendly (XGBoost in `04`).


In [9]:
# Separate target and drop the ID from the feature matrix.
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])

# Label-encode every categorical (object/string) column to an integer code.
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
for c in cat_cols:
    X[c] = pd.factorize(X[c])[0]          # maps categories -> 0,1,2,...; NaN -> -1
print(f"Label-encoded {len(cat_cols)} categorical columns.")
print(f"Final feature matrix: {X.shape[1]} features, {X.shape[0]:,} rows")

Label-encoded 12 categorical columns.
Final feature matrix: 96 features, 307,511 rows


In [10]:
# Stratified 80/20 split — preserve the 8% default rate in both sets.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
print("Train:", X_train.shape, "| default rate", f"{y_train.mean()*100:.2f}%")
print("Val:  ", X_val.shape, "| default rate", f"{y_val.mean()*100:.2f}%")

# Impute missing values with the MEDIAN learned on TRAIN ONLY.
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)     # fit + transform on train
X_val_imp = imputer.transform(X_val)             # transform val with TRAIN's medians

# Scale to zero mean / unit variance, again fit on TRAIN ONLY.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)

# Back to labelled DataFrames so columns stay interpretable downstream.
feature_names = X.columns.tolist()
X_train_final = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train.index)
X_val_final = pd.DataFrame(X_val_scaled, columns=feature_names, index=X_val.index)
print("Scaling complete. No NaNs left:",
      not X_train_final.isna().any().any() and not X_val_final.isna().any().any())

Train: (246008, 96) | default rate 8.07%
Val:   (61503, 96) | default rate 8.07%


Scaling complete. No NaNs left: True


## 6 · Save processed matrices & governance checks

In [11]:
# Persist the model-ready matrices + labels (data/processed is git-ignored, regenerable).
X_train_final.to_csv(os.path.join(PROC_DIR, "X_train.csv"))
X_val_final.to_csv(os.path.join(PROC_DIR, "X_val.csv"))
y_train.to_csv(os.path.join(PROC_DIR, "y_train.csv"))
y_val.to_csv(os.path.join(PROC_DIR, "y_val.csv"))

# Persist the raw protected attributes aligned to each split (for the fairness audit in 05).
prot_indexed = protected.set_index("SK_ID_CURR")
prot_indexed.loc[df.loc[X_train.index, "SK_ID_CURR"]].to_csv(os.path.join(PROC_DIR, "protected_train.csv"))
prot_indexed.loc[df.loc[X_val.index, "SK_ID_CURR"]].to_csv(os.path.join(PROC_DIR, "protected_val.csv"))

print("Saved to", PROC_DIR, ":")
for f in sorted(os.listdir(PROC_DIR)):
    print("  ", f)

Saved to ../data/processed :
   X_train.csv
   X_val.csv
   protected_train.csv
   protected_val.csv
   y_train.csv
   y_val.csv


In [12]:
# --- Governance check 1: leakage guard ---
# Because the scaler was fit on TRAIN ONLY, train features standardize to ~0 mean by
# construction; the validation set does NOT (it uses train's statistics) — the correct,
# honest behavior. If val also had exactly 0 mean, that would signal a leak.
train_mean_check = np.allclose(X_train_final.mean().values, 0, atol=1e-6)
print("Train features standardized to ~0 mean (fit on train):", train_mean_check)

# --- Governance check 2: missingness did not differ across protected groups in a way we ignored ---
# Compare EXT_SOURCE_1 missingness by gender on the RAW joined data (before imputation).
miss_by_gender = df.assign(_miss=df["EXT_SOURCE_1"].isna()).groupby("CODE_GENDER")["_miss"].mean() * 100
print("\nEXT_SOURCE_1 missing % by gender (pre-imputation):")
print(miss_by_gender.round(1).to_string())

# --- Governance check 3: final inventory ---
print(f"\nFinal feature count: {X.shape[1]}")
print(f"  - from main table (cleaned): {X.shape[1] - total_agg_features}")
print(f"  - from auxiliary aggregations: {total_agg_features}")

Train features standardized to ~0 mean (fit on train): True

EXT_SOURCE_1 missing % by gender (pre-imputation):
CODE_GENDER
F    54.7
M    59.6

Final feature count: 96
  - from main table (cleaned): 59
  - from auxiliary aggregations: 37


**Governance notes.**
- **Leakage:** imputer and scaler were `fit` on **train only**; validation used those fitted
  statistics. Train features standardize to ~0 mean by construction — the val set does *not*,
  which is the correct, honest behavior.
- **Bias check:** `EXT_SOURCE_1` missingness is similar across genders, so median imputation
  of it does not obviously advantage one group — documented for the audit (the full check
  across all protected groups happens in `05`).
- **Label encoding caveat:** integer codes impose a false order for a *linear* model
  (LogReg); this is fine for the **tree-based XGBoost** that is our headline model. We note
  it so the model card is honest about the choice.


## 7 · Key takeaways & next step

1. Collapsed **10 raw tables** into a single matrix of **one row per applicant**.
2. Auxiliary history summarized into **37 named aggregates** (bureau, previous apps, POS,
   installments, credit card) — moderate scope, every feature documented.
3. Applied `02` fixes: sentinel → NaN + flag, dropped the 47-column housing block, collapsed
   20 document flags, engineered 4 affordability ratios.
4. **Leakage-free** preprocessing: split → impute(median) → scale, all **fit on train only**.
5. Saved `X_train / X_val / y_train / y_val` + **raw protected attributes per split** for the
   fairness audit.

---
### ➡️ Next: `04_modeling`
Train LogReg / RandomForest / **XGBoost** on the processed matrix, handle the 8% imbalance
(class weights / resampling), and tune a **cost-sensitive threshold** (a missed default costs
far more than a false alarm).
